In [ ]:
import os

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from scipy.stats import zscore, ttest_ind
from statsmodels.formula.api import ols
import statsmodels.api as sm
from statsmodels.stats.multitest import multipletests

sns.set(style="whitegrid")

In [ ]:
DATA_PATH  = "/content/FB_AD_5Choice.csv"
OUTPUT_DIR = "outputs"
RANDOM_SEED = 42

VALID_GENOTYPES = {"5XFAD", "C57BL/6J", "B6129SF2/J"}
GENDER_ORDER    = ["Male", "Female"]

FIVE_CHOICE_COLUMNS = [
    "AnimalID", "Age", "Sex", "Genotype", "Strain", "SessionName", "Date_Time",
    "AVG_Trial Analysis - Accuracy%",
    "AVG_Trial Analysis - Correct",
    "AVG_Trial Analysis - Incorrect",
    "AVG_Trial Analysis - Omission",
    "AVG_Trial Analysis - Omission%",
    "AVG_Trial Analysis - Premature",
    "AVG_Perseverative Correct - Total",
    "AVG_Perseverative Incorrect - Total",
    "AVG_Trial Analysis - Correct Response Latency",
    "AVG_Trial Analysis - Incorrect Response Latency",
    "AVG_Trial Analysis - Reward Collection Latency",
    "AVG_Threshold - Accuracy %",
    "AVG_Threshold - Omission %",
    "AVG_Threshold - Trials",
    "AVG_Threshold - Condition",
]

METRICS = [
    "AVG_Trial Analysis - Accuracy%",
    "AVG_Trial Analysis - Omission%",
    "AVG_Trial Analysis - Correct Response Latency",
    "AVG_Trial Analysis - Incorrect Response Latency",
    "AVG_Perseverative Correct - Total",
    "AVG_Threshold - Accuracy %",
    "AVG_Threshold - Condition",
    "AVG_Trial Analysis - Premature",
    "AVG_Trial Analysis - Reward Collection Latency",
]

METRIC_DISPLAY_ORDER = [
    "Average Total Perseverative Correct Responses",
    "Average Accuracy At Threshold",
    "Average Task Condition At Threshold",
    "Average Trial Accuracy",
    "Average Latency For Correct Responses",
    "Average Latency For Incorrect Responses",
    "Average Omission Rate",
    "Average Number Of Premature Responses",
    "Average Latency To Collect Reward",
]

os.makedirs(OUTPUT_DIR, exist_ok=True)

In [ ]:
def out(filename: str) -> str:
    return os.path.join(OUTPUT_DIR, filename)


def save_fig(fig: plt.Figure, filename: str, dpi: int = 300) -> None:
    path = out(filename)
    fig.savefig(path, dpi=dpi, bbox_inches="tight")
    plt.close(fig)
    print(f"Saved figure: {path}")


def save_csv(df: pd.DataFrame, filename: str) -> None:
    path = out(filename)
    df.to_csv(path, index=False)
    print(f"Saved CSV:    {path}")


def clean_metric_name(name: str) -> str:
    mapping = {
        "AVG_Perseverative Correct - Total":              "Average Total Perseverative Correct Responses",
        "AVG_Threshold - Accuracy %":                     "Average Accuracy At Threshold",
        "AVG_Threshold - Condition":                      "Average Task Condition At Threshold",
        "AVG_Trial Analysis - Accuracy%":                 "Average Trial Accuracy",
        "AVG_Trial Analysis - Correct Response Latency":  "Average Latency For Correct Responses",
        "AVG_Trial Analysis - Incorrect Response Latency":"Average Latency For Incorrect Responses",
        "AVG_Trial Analysis - Omission%":                 "Average Omission Rate",
        "AVG_Trial Analysis - Premature":                 "Average Number Of Premature Responses",
        "AVG_Trial Analysis - Reward Collection Latency": "Average Latency To Collect Reward",
    }
    return mapping.get(name, name.replace("_", " ").replace("-", " ").title())


def cohen_d(x: pd.Series, y: pd.Series) -> float:
    nx, ny = len(x), len(y)
    pooled_std = np.sqrt(
        ((nx - 1) * x.std(ddof=1) ** 2 + (ny - 1) * y.std(ddof=1) ** 2)
        / (nx + ny - 2)
    )
    return (x.mean() - y.mean()) / pooled_std


def remove_outliers(df: pd.DataFrame, cols: list, threshold: float = 3.0) -> pd.DataFrame:
    z = df[cols].apply(zscore)
    return df[(z.abs() <= threshold).all(axis=1)]


In [ ]:
def load_and_clean(path: str, columns: list, metrics: list) -> pd.DataFrame:
    df = pd.read_csv(path)
    df = df[df["Genotype"].isin(VALID_GENOTYPES)][columns].copy()

    SEX_MAP = {"M": "Male", "F": "Female", "m": "Male", "f": "Female", "male": "Male", "female": "Female"}
    df["Sex"] = df["Sex"].str.strip().map(lambda v: SEX_MAP.get(v, v))
    df["DiseaseStatus"] = df["Genotype"].map(
        lambda g: "AD" if g == "5XFAD" else "Control"
    )

    df = df.dropna(subset=metrics, how="any")
    df = remove_outliers(df, metrics)
    return df

In [ ]:
def plot_grouped_boxplot(df, metric: str, filename: str) -> None:
    clean = clean_metric_name(metric)
    fig, ax = plt.subplots(figsize=(10, 6))
    sns.boxplot(
        data=df, x="Sex", y=metric, hue="DiseaseStatus",
        order=GENDER_ORDER, dodge=True, showfliers=False,
        palette="Set2", ax=ax,
    )
    ax.set(
        title=f"Boxplot of {clean} by Sex and Disease Status",
        xlabel="Sex", ylabel=clean,
    )
    ax.legend(title="Disease Status", fontsize=10, title_fontsize=12)
    plt.tight_layout()
    save_fig(fig, filename)


def plot_grouped_barplot(df, metric: str, filename: str) -> None:
    clean   = clean_metric_name(metric)
    avg_df  = df.groupby(["Sex", "DiseaseStatus"])[metric].mean().reset_index()
    fig, ax = plt.subplots(figsize=(10, 6))
    sns.barplot(
        data=avg_df, x="Sex", y=metric, hue="DiseaseStatus",
        order=GENDER_ORDER, palette="Set2", ax=ax,
    )
    ax.set(
        title=f"{clean} by Sex and Disease Status",
        xlabel="Sex", ylabel=clean,
    )
    ax.legend(title="Disease Status", fontsize=10, title_fontsize=12)
    plt.tight_layout()
    save_fig(fig, filename)

In [ ]:
def run_ttests_by_sex(df: pd.DataFrame, metrics: list) -> pd.DataFrame:
    rows = []
    for sex in GENDER_ORDER:
        sex_df = df[df["Sex"] == sex]
        for metric in metrics:
            ad_vals   = sex_df[sex_df["DiseaseStatus"] == "AD"][metric].dropna()
            ctrl_vals = sex_df[sex_df["DiseaseStatus"] == "Control"][metric].dropna()

            if len(ad_vals) < 2 or len(ctrl_vals) < 2:
                continue

            t_stat, p_raw = ttest_ind(ad_vals, ctrl_vals, equal_var=False)
            rows.append({
                "gender":       sex,
                "metric":       metric,
                "metric_clean": clean_metric_name(metric),
                "t_stat":       t_stat,
                "p_raw":        p_raw,
                "n_ad":         len(ad_vals),
                "mean_ad":      ad_vals.mean(),
                "std_ad":       ad_vals.std(),
                "n_control":    len(ctrl_vals),
                "mean_control": ctrl_vals.mean(),
                "std_control":  ctrl_vals.std(),
            })

    result_df = pd.DataFrame(rows)
    _, result_df["p_adj"], _, _ = multipletests(result_df["p_raw"], method="bonferroni")
    result_df = result_df.sort_values(["p_adj", "metric"]).reset_index(drop=True)
    return result_df


def run_anova_group_x_sex(df: pd.DataFrame, metrics: list) -> pd.DataFrame:
    rows = []
    for metric in metrics:
        df_tmp    = df[["DiseaseStatus", "Sex", metric]].dropna()
        safe_col  = "target_metric"
        df_tmp    = df_tmp.rename(columns={metric: safe_col})

        if (
            df_tmp["DiseaseStatus"].nunique() < 2
            or df_tmp["Sex"].nunique() < 2
        ):
            print(f"Skipping ANOVA for '{metric}': insufficient group levels.")
            continue

        try:
            model      = ols(f"{safe_col} ~ C(DiseaseStatus) * C(Sex)", data=df_tmp).fit()
            anova_tbl  = sm.stats.anova_lm(model, typ=2)

            rows.append({
                "metric":            metric,
                "metric_clean":      clean_metric_name(metric),
                "F_group":           anova_tbl.loc["C(DiseaseStatus)", "F"],
                "p_group_raw":       anova_tbl.loc["C(DiseaseStatus)", "PR(>F)"],
                "F_sex":             anova_tbl.loc["C(Sex)", "F"],
                "p_sex_raw":         anova_tbl.loc["C(Sex)", "PR(>F)"],
                "F_interaction":     anova_tbl.loc["C(DiseaseStatus):C(Sex)", "F"],
                "p_interaction_raw": anova_tbl.loc["C(DiseaseStatus):C(Sex)", "PR(>F)"],
            })
        except Exception as e:
            print(f"ANOVA failed for '{metric}': {e}")

    anova_df = pd.DataFrame(rows)

    for factor in ["group", "sex", "interaction"]:
        _, anova_df[f"p_{factor}_adj"], _, _ = multipletests(
            anova_df[f"p_{factor}_raw"], method="bonferroni"
        )

    return anova_df.sort_values("p_interaction_adj").reset_index(drop=True)


def build_summary_table(df: pd.DataFrame, metrics: list) -> pd.DataFrame:

    rows = []
    for metric in metrics:
        for sex in GENDER_ORDER:
            ctrl_vals = df[(df["DiseaseStatus"] == "Control") & (df["Sex"] == sex)][metric].dropna()
            ad_vals   = df[(df["DiseaseStatus"] == "AD")      & (df["Sex"] == sex)][metric].dropna()

            if len(ctrl_vals) == 0 or len(ad_vals) == 0:
                continue

            t_stat, p_raw  = ttest_ind(ad_vals, ctrl_vals, equal_var=False)
            pct_change      = (ad_vals.mean() - ctrl_vals.mean()) / ctrl_vals.mean() * 100

            rows.append({
                "Metric":                clean_metric_name(metric),
                "Sex":                   sex,
                "Control Mean ± SD":     f"{ctrl_vals.mean():.2f} ± {ctrl_vals.std():.2f}",
                "AD Mean ± SD":          f"{ad_vals.mean():.2f} ± {ad_vals.std():.2f}",
                "% Change from Control": round(pct_change, 2),
                "t-stat":                round(t_stat, 4),
                "_p_raw":                p_raw,
                "Cohen's d":             round(cohen_d(ad_vals, ctrl_vals), 4),
            })

    summary_df = pd.DataFrame(rows)
    _, summary_df["Adjusted p-value (Bonferroni)"], _, _ = multipletests(
        summary_df["_p_raw"], method="bonferroni"
    )
    summary_df = summary_df.rename(columns={"_p_raw": "p-value"}).copy()

    for col in ["p-value", "Adjusted p-value (Bonferroni)"]:
        summary_df[col] = summary_df[col].apply(
            lambda x: f"{x:.4e}" if x < 0.001 else f"{x:.6f}"
        )

    return summary_df[[
        "Metric", "Sex", "Control Mean ± SD", "AD Mean ± SD",
        "% Change from Control", "t-stat", "p-value",
        "Adjusted p-value (Bonferroni)", "Cohen's d",
    ]]

In [ ]:
def plot_ttest_heatmap(ttest_df: pd.DataFrame) -> None:
    pval_matrix = (
        ttest_df
        .pivot(index="metric_clean", columns="gender", values="p_adj")
        .reindex(METRIC_DISPLAY_ORDER)
    )
    log_pvals = -np.log10(pval_matrix.replace(0, 1e-300))

    fig, ax = plt.subplots(figsize=(12, 8))
    sns.heatmap(
        log_pvals, annot=True, fmt=".3f", cmap="coolwarm",
        cbar_kws={"label": r"$-\log_{10}(\text{adj. p-value})$"}, ax=ax,
    )
    ax.set(
        title="T-test: AD vs. Control by Sex and Metric (Bonferroni Corrected)",
        xlabel="Sex", ylabel="Metric",
    )
    ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha="right")
    ax.set_yticklabels(ax.get_yticklabels(), rotation=0)
    plt.tight_layout()
    save_fig(fig, "ttest_sex_pvalue_heatmap.png")


In [ ]:
def plot_anova_barplot(anova_df: pd.DataFrame) -> None:
    metric_order_raw = (
        anova_df
        .set_index("metric")[["p_group_adj", "p_sex_adj", "p_interaction_adj"]]
        .min(axis=1)
        .sort_values()
        .index.tolist()
    )
    metric_order_clean = [clean_metric_name(m) for m in metric_order_raw]

    melted = pd.melt(
        anova_df,
        id_vars=["metric"],
        value_vars=["p_group_adj", "p_sex_adj", "p_interaction_adj"],
        var_name="factor",
        value_name="p_adj",
    )
    melted["factor"] = melted["factor"].replace({
        "p_group_adj":       "Group",
        "p_sex_adj":         "Sex",
        "p_interaction_adj": "Interaction",
    })
    melted["-log10(p_adj)"] = -np.log10(melted["p_adj"].replace(0, 1e-300))
    melted["metric_clean"]  = melted["metric"].apply(clean_metric_name)
    melted["metric_clean"]  = pd.Categorical(
        melted["metric_clean"], categories=metric_order_clean, ordered=True
    )

    fig, ax = plt.subplots(figsize=(14, 8))
    sns.barplot(
        data=melted, x="metric_clean", y="-log10(p_adj)",
        hue="factor", palette="muted", ax=ax,
    )
    ax.axhline(
        -np.log10(0.05), color="red", linestyle="--", label="p = 0.05 threshold"
    )
    ax.set(
        title="ANOVA Significance: Group, Sex, and Interaction\n(Bonferroni Corrected)",
        xlabel="Metric",
        ylabel=r"$-\log_{10}(\text{adj. p-value})$",
    )
    ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha="right")
    ax.legend(title="ANOVA Factor", loc="upper right")
    plt.tight_layout()
    save_fig(fig, "anova_significance_barplot.png")

In [ ]:
def main() -> None:
    np.random.seed(RANDOM_SEED)

    df = load_and_clean(DATA_PATH, FIVE_CHOICE_COLUMNS, METRICS)

    print("Cohort summary after preprocessing:")
    print(df["Genotype"].value_counts(), "\n")
    print(pd.crosstab(df["Sex"], df["DiseaseStatus"]))

    save_csv(df, "five_choice_cleaned.csv")

    for metric in METRICS:
        slug = metric.replace(" ", "_").replace("%", "pct").replace("/", "_")
        plot_grouped_boxplot(df, metric, f"boxplot_sex_{slug}.png")
        plot_grouped_barplot(df, metric, f"barplot_sex_{slug}.png")

    ttest_df = run_ttests_by_sex(df, METRICS)
    save_csv(ttest_df, "ttest_by_sex.csv")
    plot_ttest_heatmap(ttest_df)

    anova_df = run_anova_group_x_sex(df, METRICS)
    save_csv(anova_df, "anova_group_x_sex.csv")
    plot_anova_barplot(anova_df)

    summary_df = build_summary_table(df, METRICS)
    save_csv(summary_df, "summary_ad_vs_control_by_sex.csv")

    print(f"\nAll outputs saved to: {os.path.abspath(OUTPUT_DIR)}/")

if __name__ == "__main__":
    main()